# 📱 手机传感器技术 —— Python 演示程序集

> 本 Notebook 汇总了 [手机传感器技术](https://zebedee2021.github.io/Mobile-Sensor-2026/) 课程网站中的所有 Python 代码示例，可在 Google Colab 中直接运行。

**内容组织:**
1. 环境准备
2. 传感器基础算法 (加速度计、陀螺仪、磁力计、气压计)
3. 数据采集实验 (计步器、指南针、楼层检测、手势识别)
4. SensorLog 数据分析
5. 数据上云服务端 (参考代码)

---
## 1. 环境准备

运行下方单元格安装所需依赖。Colab 已预装 numpy、pandas、matplotlib、scipy、scikit-learn。

> **中文显示**: Colab 默认无中文字体，下方代码会自动安装 Noto Sans CJK 字体。首次运行需要几秒钟。

In [ ]:
# Colab 已预装: numpy, pandas, matplotlib, scipy, scikit-learn
# 以下为服务端参考代码所需，在 Colab 中可不安装
# !pip install flask paho-mqtt dash plotly -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

# ============================================================
# Colab 中文字体支持
# Colab 默认无中文字体，需安装 Noto Sans CJK
# ============================================================
import subprocess, sys, glob
from matplotlib.font_manager import fontManager

if 'google.colab' in sys.modules:
    subprocess.run('apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1',
                   shell=True)
    # 将新安装的字体注册到 matplotlib
    for font_file in glob.glob('/usr/share/fonts/**/Noto*CJK*', recursive=True):
        fontManager.addfont(font_file)
    print('已安装 Noto Sans CJK 中文字体')

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 验证中文显示
fig, ax = plt.subplots(figsize=(4, 1))
ax.text(0.5, 0.5, '中文显示测试 ✓', ha='center', va='center', fontsize=16)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.show()

print('环境准备完成!')

---
## 2. 传感器基础算法

本节演示各类传感器的基础算法，使用合成数据即可运行。

### 2.1 加速度计: 屏幕方向检测

通过检测重力加速度在 X、Y 轴上的分量判断手机朝向。

In [ ]:
import math

def detect_orientation(ax, ay, az):
    """根据加速度计数据判断屏幕方向"""
    angle = math.atan2(ay, ax) * 180 / math.pi

    if 45 < angle < 135:
        return "竖屏正向 (Portrait)"
    elif -135 < angle < -45:
        return "竖屏反向 (Portrait Inverted)"
    elif -45 < angle < 45:
        return "横屏右转 (Landscape Right)"
    else:
        return "横屏左转 (Landscape Left)"

# 测试: 模拟不同姿态下的加速度值
test_cases = [
    (0.0, 9.8, 0.0,   "手机竖直正放"),
    (0.0, -9.8, 0.0,  "手机倒置"),
    (9.8, 0.0, 0.0,   "手机向右横放"),
    (-9.8, 0.0, 0.0,  "手机向左横放"),
    (0.0, 0.0, -9.8,  "手机平放屏幕朝上"),
]

print("加速度计屏幕方向检测演示")
print("=" * 50)
for ax, ay, az, desc in test_cases:
    result = detect_orientation(ax, ay, az)
    print(f"{desc:15s} | ax={ax:6.1f}, ay={ay:6.1f} | → {result}")

### 2.2 加速度计: 简易计步器

利用加速度合成量的周期性波峰检测步态。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def compute_magnitude(ax, ay, az):
    """计算加速度合成量"""
    return math.sqrt(ax**2 + ay**2 + az**2)

def simple_step_counter(data, threshold=10.5, min_interval=0.3):
    """
    简易峰值检测计步
    data: [(timestamp, ax, ay, az), ...]
    """
    steps = 0
    last_step_time = 0

    for t, ax, ay, az in data:
        mag = compute_magnitude(ax, ay, az)
        if mag > threshold and (t - last_step_time) > min_interval:
            steps += 1
            last_step_time = t

    return steps

# 生成模拟步行数据 (100步, ~2Hz 步频)
np.random.seed(42)
fs = 50  # 采样率 50Hz
duration = 50  # 50秒
t = np.arange(0, duration, 1/fs)

# 重力基线 (9.8 m/s²) + 步行波形 (~2Hz 正弦波) + 噪声
step_freq = 2.0  # Hz
az_sim = 9.8 + 1.5 * np.sin(2 * np.pi * step_freq * t) + np.random.normal(0, 0.3, len(t))
ax_sim = np.random.normal(0, 0.5, len(t))
ay_sim = np.random.normal(0, 0.5, len(t))
mag = np.sqrt(ax_sim**2 + ay_sim**2 + az_sim**2)

# 计步
data = list(zip(t, ax_sim, ay_sim, az_sim))
steps = simple_step_counter(data, threshold=10.5, min_interval=0.3)
print(f"模拟时长: {duration}秒, 步频: {step_freq}Hz")
print(f"理论步数: {int(duration * step_freq)}")
print(f"检测步数: {steps}")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t[:250], az_sim[:250], linewidth=0.8)
axes[0].set_ylabel('Z轴加速度 (m/s²)')
axes[0].set_title('模拟步行加速度数据 (前5秒)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:250], mag[:250], linewidth=0.8, color='orange')
axes[1].axhline(y=10.5, color='r', linestyle='--', label='阈值=10.5')
axes[1].set_ylabel('合成加速度 (m/s²)')
axes[1].set_xlabel('时间 (s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 陀螺仪: 电子图像稳定 (EIS)

根据陀螺仪角速度数据计算补偿旋转，实现电子防抖。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def eis_compensation(frame_rotation, gyro_data, dt):
    """
    电子防抖: 根据陀螺仪数据计算补偿旋转
    gyro_data: (gx, gy, gz) rad/s
    dt: 采样间隔 (s)
    """
    # 角度增量 = 角速度 × 时间
    d_roll  = gyro_data[0] * dt
    d_pitch = gyro_data[1] * dt
    d_yaw   = gyro_data[2] * dt

    # 低通滤波平滑 (简化版)
    alpha = 0.95
    smoothed_roll  = alpha * frame_rotation[0] + (1 - alpha) * d_roll
    smoothed_pitch = alpha * frame_rotation[1] + (1 - alpha) * d_pitch

    return (smoothed_roll, smoothed_pitch, d_yaw)

# 演示: 模拟手持拍摄时的抖动
np.random.seed(42)
dt = 1 / 30  # 30fps
n_frames = 150  # 5秒

# 模拟陀螺仪抖动数据 (rad/s)
gyro_roll  = np.random.normal(0, 0.1, n_frames)
gyro_pitch = np.random.normal(0, 0.08, n_frames)
gyro_yaw   = np.random.normal(0, 0.05, n_frames)

# 累积原始旋转和补偿后旋转
raw_angles = []
comp_angles = []
raw_rot = (0, 0, 0)
comp_rot = (0, 0, 0)

for i in range(n_frames):
    gyro = (gyro_roll[i], gyro_pitch[i], gyro_yaw[i])
    raw_rot = (raw_rot[0] + gyro[0]*dt, raw_rot[1] + gyro[1]*dt, raw_rot[2] + gyro[2]*dt)
    comp_rot = eis_compensation(comp_rot, gyro, dt)
    raw_angles.append(math.degrees(raw_rot[0]))
    comp_angles.append(math.degrees(comp_rot[0]))

t = np.arange(n_frames) * dt
plt.figure(figsize=(12, 4))
plt.plot(t, raw_angles, label='原始旋转 (roll)', alpha=0.8)
plt.plot(t, comp_angles, label='EIS 补偿后', linewidth=2)
plt.xlabel('时间 (s)')
plt.ylabel('角度 (°)')
plt.title('电子图像稳定 (EIS) 演示')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.4 陀螺仪: 互补滤波器

融合加速度计 (低频可信) 和陀螺仪 (高频可信)，计算稳定的姿态角。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

class ComplementaryFilter:
    """互补滤波器: 融合加速度计 (低频可信) 和陀螺仪 (高频可信)"""

    def __init__(self, alpha=0.98):
        self.alpha = alpha
        self.pitch = 0.0
        self.roll = 0.0

    def update(self, ax, ay, az, gx, gy, gz, dt):
        # 加速度计估计的倾角
        accel_pitch = math.atan2(ax, math.sqrt(ay**2 + az**2))
        accel_roll  = math.atan2(ay, math.sqrt(ax**2 + az**2))

        # 互补滤波
        self.pitch = self.alpha * (self.pitch + gx * dt) + (1 - self.alpha) * accel_pitch
        self.roll  = self.alpha * (self.roll  + gy * dt) + (1 - self.alpha) * accel_roll

        return self.pitch, self.roll

# 演示: 模拟带漂移的陀螺仪和带噪声的加速度计
np.random.seed(0)
dt = 0.02  # 50Hz
n = 500    # 10秒
t = np.arange(n) * dt

# 真实倾斜角: 缓慢摆动
true_pitch = 0.3 * np.sin(2 * np.pi * 0.2 * t)

# 加速度计: 有噪声但无漂移
accel_pitch = true_pitch + np.random.normal(0, 0.15, n)

# 陀螺仪: 无噪声但有漂移
gyro_rate = np.gradient(true_pitch, dt) + 0.01  # 加入常值偏移

# 纯陀螺仪积分 (有漂移)
gyro_only = np.cumsum(gyro_rate * dt)

# 互补滤波
cf = ComplementaryFilter(alpha=0.98)
cf_pitch = []
for i in range(n):
    ax_i = math.sin(accel_pitch[i]) * 9.8
    az_i = math.cos(accel_pitch[i]) * 9.8
    p, _ = cf.update(ax_i, 0, az_i, gyro_rate[i], 0, 0, dt)
    cf_pitch.append(p)

plt.figure(figsize=(12, 5))
plt.plot(t, true_pitch, 'k-', label='真实值', linewidth=2)
plt.plot(t, accel_pitch, 'b-', label='加速度计 (带噪声)', alpha=0.4)
plt.plot(t, gyro_only, 'r--', label='纯陀螺仪积分 (有漂移)', alpha=0.7)
plt.plot(t, cf_pitch, 'g-', label='互补滤波 (α=0.98)', linewidth=2)
plt.xlabel('时间 (s)')
plt.ylabel('Pitch 角 (rad)')
plt.title('互补滤波器: 融合加速度计与陀螺仪')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.5 磁力计: 椭球拟合标定

手机内部永磁材料会产生硬铁干扰，通过"8"字标定消除偏置。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calibrate_magnetometer(raw_data):
    """
    椭球拟合标定 (简化版)
    raw_data: Nx3 数组 [(mx, my, mz), ...]
    返回: offset (硬铁偏置), scale (软铁校正)
    """
    # 求各轴最大最小值的中点作为偏置估计
    offset = np.mean([np.max(raw_data, axis=0),
                      np.min(raw_data, axis=0)], axis=0)

    # 求各轴范围作为比例因子
    ranges = np.max(raw_data, axis=0) - np.min(raw_data, axis=0)
    avg_range = np.mean(ranges)
    scale = avg_range / ranges

    return offset, scale

def apply_calibration(raw, offset, scale):
    """应用标定参数"""
    return (raw - offset) * scale

# 模拟"8"字标定数据: 球面 + 偏置 + 椭圆化
np.random.seed(42)
n_points = 500
theta = np.random.uniform(0, 2 * np.pi, n_points)
phi = np.random.uniform(-np.pi/2, np.pi/2, n_points)

# 真实球面 (30 μT 半径) + 硬铁偏置 + 软铁拉伸
hard_iron = np.array([15, -10, 5])    # 硬铁偏置
soft_iron = np.array([1.2, 0.9, 1.0]) # 软铁拉伸
r = 30  # 地磁场强度 (μT)

raw_data = np.column_stack([
    r * np.cos(phi) * np.cos(theta) * soft_iron[0] + hard_iron[0],
    r * np.cos(phi) * np.sin(theta) * soft_iron[1] + hard_iron[1],
    r * np.sin(phi) * soft_iron[2] + hard_iron[2]
]) + np.random.normal(0, 1, (n_points, 3))  # 加噪声

# 标定
offset, scale = calibrate_magnetometer(raw_data)
calibrated = apply_calibration(raw_data, offset, scale)

print(f"硬铁偏置 (offset): [{offset[0]:.1f}, {offset[1]:.1f}, {offset[2]:.1f}] μT")
print(f"软铁校正 (scale):  [{scale[0]:.3f}, {scale[1]:.3f}, {scale[2]:.3f}]")
print(f"真实偏置:         [{hard_iron[0]}, {hard_iron[1]}, {hard_iron[2]}] μT")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(raw_data[:, 0], raw_data[:, 1], s=3, alpha=0.5)
axes[0].set_title('标定前 (XY平面)')
axes[0].set_xlabel('Mx (μT)')
axes[0].set_ylabel('My (μT)')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

axes[1].scatter(calibrated[:, 0], calibrated[:, 1], s=3, alpha=0.5, color='green')
axes[1].set_title('标定后 (XY平面)')
axes[1].set_xlabel('Mx (μT)')
axes[1].set_ylabel('My (μT)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

plt.suptitle('磁力计椭球拟合标定效果')
plt.tight_layout()
plt.show()

### 2.6 磁力计: 电子指南针 (倾斜补偿)

利用加速度计数据进行倾斜补偿，计算磁航向角。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def compass_heading(mx, my, mz, pitch, roll):
    """
    计算磁航向角 (需要倾斜补偿)
    mx, my, mz: 磁力计读数 (μT)
    pitch, roll: 由加速度计得到的倾斜角 (rad)
    """
    # 倾斜补偿
    mx_comp = (mx * math.cos(pitch)
               + mz * math.sin(pitch))
    my_comp = (mx * math.sin(roll) * math.sin(pitch)
               + my * math.cos(roll)
               - mz * math.sin(roll) * math.cos(pitch))

    # 计算航向角
    heading = math.atan2(-my_comp, mx_comp)
    heading_deg = math.degrees(heading)

    if heading_deg < 0:
        heading_deg += 360

    return heading_deg

# 演示: 旋转一周的指南针
n = 360
angles = np.linspace(0, 360, n, endpoint=False)
headings = []

Bh = 30.0  # 水平磁场分量 (μT), 约北京地区
Bv = -45.0 # 垂直磁场分量 (北半球向下)

for a in angles:
    rad = math.radians(a)
    mx = Bh * math.cos(rad)
    my = Bh * math.sin(rad)
    mz = Bv
    h = compass_heading(mx, my, mz, pitch=0, roll=0)
    headings.append(h)

# 测试带倾斜的情况
print("电子指南针演示")
print("=" * 50)
test_tilts = [(0, 0, "水平"), (0.3, 0, "pitch=17°"), (0, 0.3, "roll=17°")]
for pitch, roll, desc in test_tilts:
    h = compass_heading(Bh, 0, Bv, pitch, roll)
    print(f"姿态: {desc:12s} | 航向: {h:.1f}° (应接近0°/360°)")

# 绘制极坐标图
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, polar=True)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.plot(np.radians(angles), headings, 'b.', markersize=2)
ax.plot(np.radians(angles), angles, 'r--', alpha=0.5, label='理想值')
ax.set_title('指南针航向角 vs 实际方位', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.show()

### 2.7 气压计: 气压转海拔 & 楼层检测

利用气压高度公式将气压转化为海拔，并检测楼层变化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pressure_to_altitude(pressure, sea_level_pressure=1013.25):
    """将气压转换为海拔高度 (m)"""
    return 44330.0 * (1.0 - (pressure / sea_level_pressure) ** 0.1903)

def detect_floor_change(p1, p2, floor_height=3.0):
    """检测楼层变化"""
    h1 = pressure_to_altitude(p1)
    h2 = pressure_to_altitude(p2)
    delta_floors = (h2 - h1) / floor_height
    return round(delta_floors)

# 演示 1: 气压与海拔关系
pressures = np.linspace(300, 1013.25, 200)
altitudes = [pressure_to_altitude(p) for p in pressures]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(pressures, altitudes)
plt.xlabel('气压 (hPa)')
plt.ylabel('海拔 (m)')
plt.title('气压与海拔的关系')
plt.grid(True, alpha=0.3)

# 演示 2: 模拟上楼过程
base_pressure = 1013.25
floors_data = []
for floor in range(1, 11):
    p = base_pressure - (floor - 1) * 0.36
    h = pressure_to_altitude(p) - pressure_to_altitude(base_pressure)
    detected = detect_floor_change(base_pressure, p)
    floors_data.append((floor, p, h, detected + 1))

plt.subplot(1, 2, 2)
real_floors = [f[0] for f in floors_data]
detected_floors = [f[3] for f in floors_data]
plt.plot(real_floors, real_floors, 'r--', label='真实楼层')
plt.plot(real_floors, detected_floors, 'bo-', label='检测楼层')
plt.xlabel('实际楼层')
plt.ylabel('检测楼层')
plt.title('楼层检测精度')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印详细数据
print("\n楼层检测详细数据:")
print(f"{'楼层':<6s}{'气压(hPa)':<12s}{'相对高度(m)':<14s}{'检测楼层':<10s}")
for floor, p, h, det in floors_data:
    print(f"{floor:<6d}{p:<12.2f}{h:<14.2f}{det:<10d}")

---
## 3. 数据采集实验

本节演示完整的数据分析流程，使用合成数据模拟实际采集场景。

> 💡 实际使用时，将合成数据替换为 SensorLog / Sensor Logger 导出的 CSV 文件即可。

### 3.1 实验一: 计步器 (带通滤波 + 峰值检测)

使用 scipy 的带通滤波和峰值检测实现更精确的计步算法。

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks, butter, filtfilt
import matplotlib.pyplot as plt

# 生成模拟数据 (代替实际 CSV)
np.random.seed(123)
fs = 50  # 采样率 50Hz
duration = 50  # 50秒
n_samples = fs * duration
t = np.arange(n_samples) / fs

# 模拟 100 步的加速度数据
step_freq = 2.0  # 2Hz 步频
noise = np.random.normal(0, 0.3, n_samples)
df = pd.DataFrame({
    'accelerometerAccelerationX': np.random.normal(0, 0.5, n_samples),
    'accelerometerAccelerationY': np.random.normal(0, 0.3, n_samples) + 0.5 * np.sin(2*np.pi*step_freq*t + 0.5),
    'accelerometerAccelerationZ': 9.8 + 1.5 * np.sin(2*np.pi*step_freq*t) + noise,
})

# 计算加速度合成量
df['magnitude'] = np.sqrt(
    df['accelerometerAccelerationX']**2 +
    df['accelerometerAccelerationY']**2 +
    df['accelerometerAccelerationZ']**2
)

# 带通滤波 (保留步行频率 1-3 Hz)
b, a = butter(4, [1, 3], btype='band', fs=fs)
df['filtered'] = filtfilt(b, a, df['magnitude'])

# 峰值检测
peaks, properties = find_peaks(
    df['filtered'],
    height=0.2,           # 最小峰值高度
    distance=fs * 0.3     # 最小峰间距 (0.3s, 对应 ~200 steps/min)
)

print(f"检测到步数: {len(peaks)}")
print(f"理论步数: {int(duration * step_freq)}")
print(f"误差: {abs(len(peaks) - 100)}步")

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(t[:500], df['magnitude'].values[:500], linewidth=0.5)
axes[0].set_ylabel('原始合成加速度 (m/s²)')
axes[0].set_title('计步器算法演示')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:500], df['filtered'].values[:500], linewidth=0.8, color='orange')
peak_mask = peaks[peaks < 500]
axes[1].plot(t[peak_mask], df['filtered'].values[peak_mask], 'rv', markersize=6, label='检测到的步伐')
axes[1].set_ylabel('滤波后 (m/s²)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 累计步数
step_times = t[peaks]
cumulative_steps = np.arange(1, len(peaks)+1)
axes[2].step(step_times, cumulative_steps, where='post')
axes[2].plot(t, t * step_freq, 'r--', alpha=0.5, label='理论步数')
axes[2].set_ylabel('累计步数')
axes[2].set_xlabel('时间 (s)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 实验二: 带倾斜补偿的电子指南针

融合加速度计和磁力计，实现抗倾斜的电子指南针。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def tilt_compensated_heading(ax, ay, az, mx, my, mz):
    """带倾斜补偿的航向角计算"""
    # 归一化加速度
    norm_a = np.sqrt(ax**2 + ay**2 + az**2)
    ax, ay, az = ax/norm_a, ay/norm_a, az/norm_a

    # 计算倾斜角
    pitch = np.arcsin(-ax)
    roll = np.arcsin(ay / np.cos(pitch))

    # 倾斜补偿
    mx_comp = mx * np.cos(pitch) + mz * np.sin(pitch)
    my_comp = (mx * np.sin(roll) * np.sin(pitch)
               + my * np.cos(roll)
               - mz * np.sin(roll) * np.cos(pitch))

    # 航向角
    heading = np.degrees(np.arctan2(-my_comp, mx_comp))
    if heading < 0:
        heading += 360
    return heading

# 测试不同倾斜角下的航向精度
print("电子指南针 —— 倾斜补偿对比")
print("=" * 60)

# 模拟北京地区地磁场 (Bh≈30μT, Bv≈-45μT)
Bh, Bv = 30.0, -45.0

test_configs = [
    (0.01, -0.02, -1.0, Bh, 0, Bv, "水平放置, 朝北"),
    (0.01, -0.02, -1.0, 0, Bh, Bv, "水平放置, 朝东"),
    (-0.3, -0.02, -0.95, Bh, 0, Bv, "前倾 17°, 朝北"),
    (0.01, 0.3, -0.95, Bh, 0, Bv, "右倾 17°, 朝北"),
]

for ax, ay, az, mx, my, mz, desc in test_configs:
    heading = tilt_compensated_heading(ax, ay, az, mx, my, mz)
    heading_no_comp = np.degrees(np.arctan2(-my, mx))
    if heading_no_comp < 0:
        heading_no_comp += 360
    print(f"{desc:20s} | 有补偿: {heading:6.1f}° | 无补偿: {heading_no_comp:6.1f}°")

### 3.3 实验三: 气压计测楼层

模拟从 1 楼到 5 楼的气压变化，计算相对高度和楼层。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def pressure_to_altitude(p, p0=1013.25):
    return 44330 * (1 - (p / p0) ** 0.1903)

# 模拟乘电梯从 1楼到 5楼, 每层停 10秒
np.random.seed(42)
fs = 10  # 10 Hz
base_p = 1013.25
floor_dp = 0.36  # 每层气压差 hPa

segments = []
for floor in range(5):
    # 停留阶段 (10秒)
    n_stay = fs * 10
    p_stay = base_p - floor * floor_dp + np.random.normal(0, 0.02, n_stay)
    segments.append(p_stay)
    
    if floor < 4:
        # 上升阶段 (5秒)
        n_rise = fs * 5
        p_rise = np.linspace(base_p - floor * floor_dp, 
                             base_p - (floor+1) * floor_dp, n_rise)
        p_rise += np.random.normal(0, 0.02, n_rise)
        segments.append(p_rise)

pressure = np.concatenate(segments)
t = np.arange(len(pressure)) / fs

df = pd.DataFrame({'pressure': pressure})
df['altitude'] = df['pressure'].apply(pressure_to_altitude)
df['relative_alt'] = df['altitude'] - df['altitude'].iloc[0]
df['floor'] = (df['relative_alt'] / 3.0).round().astype(int) + 1

print(f"起始气压: {df['pressure'].iloc[0]:.2f} hPa")
print(f"终止气压: {df['pressure'].iloc[-1]:.2f} hPa")
print(f"气压差: {df['pressure'].iloc[-1] - df['pressure'].iloc[0]:.2f} hPa")
print(f"高度变化: {df['relative_alt'].iloc[-1]:.1f} m")
print(f"估算楼层变化: {df['floor'].iloc[-1] - 1} 层")

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, df['pressure'], linewidth=0.8)
axes[0].set_ylabel('气压 (hPa)')
axes[0].set_title('气压计测楼层实验')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, df['relative_alt'], linewidth=0.8, color='green')
axes[1].set_ylabel('相对高度 (m)')
axes[1].grid(True, alpha=0.3)

axes[2].step(t, df['floor'], where='post', linewidth=1.5, color='red')
axes[2].set_ylabel('检测楼层')
axes[2].set_xlabel('时间 (s)')
axes[2].set_yticks(range(1, 6))
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 实验四: 手势识别 (KNN)

利用加速度计和陀螺仪数据，提取时域特征，用 KNN 分类器识别简单手势。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

def extract_features(segment_data):
    """从一个传感器数据片段中提取特征"""
    features = []
    for col in segment_data.T:  # 每个轴
        features.extend([
            col.mean(),
            col.std(),
            col.max() - col.min(),
            np.sqrt(np.mean(col**2)),  # RMS
            (np.diff(np.sign(col - col.mean())) != 0).sum(),  # 过零率
        ])
    return features

# 生成模拟手势数据
np.random.seed(42)
n_samples_per_class = 30
window_size = 50  # 1秒 @ 50Hz

X_all, y_all = [], []
gesture_names = ['摇晃', '翻转', '画圈']

for gi, gesture in enumerate(gesture_names):
    for _ in range(n_samples_per_class):
        t = np.arange(window_size) / 50.0
        if gi == 0:  # 摇晃: 高频大振幅
            data = np.column_stack([
                3 * np.sin(2*np.pi*5*t) + np.random.normal(0, 0.5, window_size),
                np.random.normal(0, 0.5, window_size),
                9.8 + np.random.normal(0, 0.3, window_size),
            ])
        elif gi == 1:  # 翻转: Z轴大幅变化
            data = np.column_stack([
                np.random.normal(0, 0.3, window_size),
                np.random.normal(0, 0.3, window_size),
                9.8 * np.cos(np.pi * t) + np.random.normal(0, 0.3, window_size),
            ])
        else:  # 画圈: X/Y轴相位差正弦
            data = np.column_stack([
                2 * np.sin(2*np.pi*2*t) + np.random.normal(0, 0.3, window_size),
                2 * np.cos(2*np.pi*2*t) + np.random.normal(0, 0.3, window_size),
                9.8 + np.random.normal(0, 0.2, window_size),
            ])
        
        features = extract_features(data)
        X_all.append(features)
        y_all.append(gi)

X_train = np.array(X_all)
y_train = np.array(y_all)

# KNN 分类
knn = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn, X_train, y_train, cv=5)
print(f"5折交叉验证准确率: {scores.mean():.2%} ± {scores.std():.2%}")
print(f"各折准确率: {[f'{s:.2%}' for s in scores]}")

# 可视化特征分布 (前两维)
plt.figure(figsize=(8, 6))
colors = ['red', 'blue', 'green']
for gi, name in enumerate(gesture_names):
    mask = y_train == gi
    plt.scatter(X_train[mask, 0], X_train[mask, 1], 
                c=colors[gi], label=name, alpha=0.6, s=40)

plt.xlabel('特征 1: X轴均值')
plt.ylabel('特征 2: X轴标准差')
plt.title('手势识别 —— 特征空间分布 (前2维)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. SensorLog 数据分析

使用 SensorLog / Sensor Logger 导出的 CSV 数据进行加载和可视化。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 生成模拟 SensorLog CSV 数据
np.random.seed(42)
n = 2500  # 50秒 @ 50Hz
t = np.arange(n) / 50.0

timestamps = pd.date_range('2026-03-27 10:00:00', periods=n, freq='20ms')

df = pd.DataFrame({
    'loggingTime': timestamps,
    'accelerometerAccelerationX': 0.02 + np.random.normal(0, 0.1, n) + 0.3 * np.sin(2*np.pi*2*t),
    'accelerometerAccelerationY': -0.98 + np.random.normal(0, 0.1, n) + 0.2 * np.sin(2*np.pi*2*t + 1),
    'accelerometerAccelerationZ': 0.05 + np.random.normal(0, 0.1, n) + 1.5 * np.sin(2*np.pi*2*t + 0.5),
})

# 如果有实际 CSV 文件, 使用以下代码加载:
# df = pd.read_csv('sensorlog_data.csv')
# df['loggingTime'] = pd.to_datetime(df['loggingTime'])

df['elapsed'] = (df['loggingTime'] - df['loggingTime'].iloc[0]).dt.total_seconds()

# 绘制加速度计数据
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for i, axis in enumerate(['X', 'Y', 'Z']):
    col = f'accelerometerAcceleration{axis}'
    if col in df.columns:
        axes[i].plot(df['elapsed'], df[col], linewidth=0.5)
        axes[i].set_ylabel(f'Accel {axis} (g)')
        axes[i].grid(True, alpha=0.3)

axes[2].set_xlabel('时间 (s)')
fig.suptitle('加速度计原始数据 (模拟 SensorLog 导出)')
plt.tight_layout()
plt.show()

# 基本统计
print("\n数据统计:")
for axis in ['X', 'Y', 'Z']:
    col = f'accelerometerAcceleration{axis}'
    print(f"  {axis}轴: 均值={df[col].mean():.4f}, 标准差={df[col].std():.4f}, "
          f"范围=[{df[col].min():.3f}, {df[col].max():.3f}]")

---
## 5. 数据上云服务端 (参考代码)

> ⚠️ 以下代码为 **服务端程序**，需要在本地或云服务器上运行，不适合在 Colab 中直接执行。请复制到本地环境使用。

### 5.1 Flask 极简接收服务

接收 Sensor Logger 通过 HTTP POST 推送的 JSON 数据，存储为 CSV。

```python
from flask import Flask, request, jsonify
import json, csv, os

app = Flask(__name__)
os.makedirs("data", exist_ok=True)

@app.route("/data", methods=["POST"])
def receive():
    data = request.get_json()
    sid = data.get("sessionId", "unknown")
    did = data.get("deviceId", "unknown")

    filepath = f"data/{sid}.csv"
    is_new = not os.path.exists(filepath)

    with open(filepath, "a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["time_ns", "device", "sensor", "x", "y", "z", "extra"])
        for item in data.get("payload", []):
            v = item.get("values", {})
            w.writerow([
                item["time"], did, item["name"],
                v.get("x", v.get("latitude", v.get("pressure", ""))),
                v.get("y", v.get("longitude", v.get("relativeAltitude", ""))),
                v.get("z", v.get("altitude", "")),
                json.dumps({k: v2 for k, v2 in v.items()
                           if k not in ("x","y","z")}, ensure_ascii=False) or ""
            ])
    return jsonify(status="ok"), 200

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)
```

**使用方法:**
1. `pip install flask`
2. `python server.py`
3. Sensor Logger Push URL 填: `http://你的IP:8000/data`

### 5.2 Plotly Dash 实时可视化仪表盘

边采集边查看加速度波形。

```python
import dash
from dash.dependencies import Output, Input
from dash import dcc, html
from datetime import datetime
import json, plotly.graph_objs as go
from collections import deque
from flask import Flask, request

server = Flask(__name__)
app = dash.Dash(__name__, server=server)

MAX_POINTS = 1000
time_q = deque(maxlen=MAX_POINTS)
ax, ay, az = deque(maxlen=MAX_POINTS), deque(maxlen=MAX_POINTS), deque(maxlen=MAX_POINTS)

app.layout = html.Div([
    html.H2("Sensor Logger 实时加速度"),
    dcc.Graph(id="live"),
    dcc.Interval(id="tick", interval=200),
])

@app.callback(Output("live", "figure"), Input("tick", "n_intervals"))
def update(_):
    traces = [go.Scatter(x=list(time_q), y=list(d), name=n)
              for d, n in zip([ax, ay, az], ["X", "Y", "Z"])]
    layout = go.Layout(xaxis={"type": "date"}, yaxis={"title": "加速度 (m/s²)"})
    if time_q:
        layout["xaxis"]["range"] = [min(time_q), max(time_q)]
    return {"data": traces, "layout": layout}

@server.route("/data", methods=["POST"])
def data():
    for d in json.loads(request.data).get("payload", []):
        if d.get("name") == "accelerometer":
            ts = datetime.fromtimestamp(d["time"] / 1e9)
            if not time_q or ts > time_q[-1]:
                time_q.append(ts)
                ax.append(d["values"]["x"])
                ay.append(d["values"]["y"])
                az.append(d["values"]["z"])
    return "ok"

if __name__ == "__main__":
    app.run_server(port=8000, host="0.0.0.0")
```

**使用方法:**
1. `pip install dash plotly flask`
2. `python dashboard.py`
3. 浏览器打开 `http://服务器IP:8000`、App Push URL 填同一地址的 `/data`

### 5.3 MQTT 订阅与存储

订阅所有设备的 MQTT 消息，写入 SQLite 数据库。

```python
import paho.mqtt.client as mqtt
import json, sqlite3
from datetime import datetime

# 初始化数据库
db = sqlite3.connect("sensor_cloud.db", check_same_thread=False)
db.execute("""CREATE TABLE IF NOT EXISTS readings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    recv_time TEXT, device TEXT, session TEXT,
    sensor TEXT, time_ns INTEGER,
    val_json TEXT
)""")

def on_connect(client, userdata, flags, rc):
    print(f"已连接 Broker, rc={rc}")
    client.subscribe("sensor/#")  # 订阅所有设备

def on_message(client, userdata, msg):
    data = json.loads(msg.payload.decode())
    device = data.get("deviceId", "")
    session = data.get("sessionId", "")
    now = datetime.utcnow().isoformat()

    for item in data.get("payload", []):
        db.execute(
            "INSERT INTO readings (recv_time,device,session,sensor,time_ns,val_json) VALUES (?,?,?,?,?,?)",
            [now, device, session, item["name"], item["time"], json.dumps(item["values"])]
        )
    db.commit()

client = mqtt.Client(transport="websockets")
client.tls_set()  # wss 需要 TLS
client.on_connect = on_connect
client.on_message = on_message
client.connect("broker.emqx.io", 8084)
print("开始监听 MQTT 消息...")
client.loop_forever()
```

**使用方法:**
1. `pip install paho-mqtt`
2. `python mqtt_subscriber.py`
3. Sensor Logger MQTT Broker 填: `wss://broker.emqx.io:8084/mqtt`, Topic: `sensor/${deviceId}`

### 5.4 批量文件上传

导出的 CSV 文件批量上传到服务器。

```python
import requests, glob, os

API = "https://your-server.com/api/upload"

for fp in glob.glob("exports/*.csv"):
    with open(fp, "rb") as f:
        r = requests.post(API, files={"file": (os.path.basename(fp), f)})
        print(f"{fp}: {r.status_code}")
```

**对应的服务端接收代码:**

```python
from flask import Flask, request
import pandas as pd, os

app = Flask(__name__)

@app.route("/api/upload", methods=["POST"])
def upload():
    f = request.files["file"]
    save_path = f"uploads/{f.filename}"
    f.save(save_path)

    # 读入 pandas 做预处理
    df = pd.read_csv(save_path)
    print(f"收到 {f.filename}: {len(df)} 行, 列: {list(df.columns)}")

    return {"status": "ok", "rows": len(df)}, 200

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
```

---
## 附录: JSON Payload 格式参考

Sensor Logger HTTP POST / MQTT 推送的 JSON 结构:

```json
{
  "messageId": 42,
  "sessionId": "a1b2c3d4-e5f6-7890",
  "deviceId": "iPhone-XYZ",
  "payload": [
    {
      "name": "accelerometer",
      "time": 1711526400000000000,
      "values": { "x": 0.023, "y": -0.981, "z": 0.045 }
    },
    {
      "name": "gyroscope",
      "time": 1711526400000000000,
      "values": { "x": 0.001, "y": -0.003, "z": 0.007 }
    }
  ]
}
```

时间戳转换: `pd.to_datetime(1711526400000000000, unit='ns')` → `2024-03-27 08:00:00 UTC`

---

**课程网站:** [https://zebedee2021.github.io/Mobile-Sensor-2026/](https://zebedee2021.github.io/Mobile-Sensor-2026/)  
**GitHub 仓库:** [https://github.com/Zebedee2021/Mobile-Sensor-2026](https://github.com/Zebedee2021/Mobile-Sensor-2026)  
**作者:** Zhiguo Zhou | 2026